# Stage 1 — Data Exploration & Preliminary Checks

Purpose: sanity-check the raw data and confirm the assumptions the pipeline is built on.  
**All production logic lives in `src/` — this notebook is exploration only. EI-map results live in `results/01_ei_maps.ipynb`; skin masking lives in `02_skin_masking.ipynb`.**

Sections:
1. Define disclosure subjects and load cubes — confirm shape (1024, 1024, 31), dtype, value range. Confirm RGB shape (1024, 1024, 3) and dtype (uint8).
2. Hyperspectral cube and bands visualization 
3. RGB vs hyperspectral alignment — confirm spatial correspondence.
4. Wavelength step confirmation — visualize all 31 bands to confirm 10 nm spacing.
5. Manifest — run `build_manifest()`, inspect 306 rows and split overrides.

In [ ]:
import sys
from pathlib import Path

# Add project root to path so src/ and config.py are importable
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import matplotlib.pyplot as plt

import config
from src.io_utils import load_cube, load_rgb, get_wavelengths, band_index_for_wavelength
from src.manifest import build_manifest

print("Imports OK")
print(f"DATA_ROOT       : {config.DATA_ROOT}")
print(f"Expected shape  : {config.CUBE_SHAPE}")
print(f"Wavelength start: {config.WAVELENGTH_START_NM} nm")
print(f"Wavelength step : {config.WAVELENGTH_STEP_NM} nm")

## 1. Define disclosure subjects

The three subjects cleared for publication are p012, p019, p027.  
All three are physically in the `train/` folder on the SSD; their manifest split is overridden to `test` via `config.SPLIT_OVERRIDE`.

In [ ]:
# All three disclosure subjects are physically in the train/ folder
DISCLOSURE_SUBJECTS = ["p012", "p019", "p027"]
SUBJECT_SPLIT_DIR = Path(config.DATA_ROOT) / "train"

cubes = {}
rgbs  = {}

for subj in DISCLOSURE_SUBJECTS:
    mat_path = SUBJECT_SPLIT_DIR / "VIS" / f"{subj}_neutral_left.mat"
    jpg_path = SUBJECT_SPLIT_DIR / "RGB" / f"{subj}_neutral_left.jpg"
    cubes[subj] = load_cube(str(mat_path))
    rgbs[subj]  = load_rgb(str(jpg_path))
    c = cubes[subj]
    print(f"{subj}  shape={c.shape}  dtype={c.dtype}  min={c.min():.4f}  max={c.max():.4f}  mean={c.mean():.4f}")

In [ ]:
import h5py
from pathlib import Path

sample_mat = SUBJECT_SPLIT_DIR / "VIS" / f"{DISCLOSURE_SUBJECTS[0]}_neutral_front.mat"
with h5py.File(str(sample_mat), "r") as f:
    all_keys = list(f.keys())
    data_keys = [k for k in all_keys if not k.startswith("#")]

print(f"All keys in .mat file : {all_keys}")
print(f"Data keys (used)      : {data_keys}")
print(f"File format           : MATLAB v7.3 / HDF5")

### .mat file structure

Inspecting the internal keys of the `.mat` file confirms what variable name was used when the dataset was saved in MATLAB, and whether any additional metadata (e.g. a wavelength vector) is stored alongside the cube. This matters because `load_cube()` auto-detects the key — if multiple variables existed, the loading logic would need to be more specific.

**Findings** — confirms the assumptions `config.py` and `src/io_utils.py` are built on:

- **Dict key:** `'cube'` — the only variable stored in each `.mat` file. Files are MATLAB v7.3 (HDF5 format), so `load_cube()` always uses the h5py loading path.
- **Value range:** calibrated reflectance (min ≈ 0.007, max = 1.000) confirmed across all three subjects — `log10(1/R)` is valid.
- **Wavelength step:** 10 nm confirmed via full 31-band strip visualization — all band index calculations in `band_index_for_wavelength()` are valid.

## 2. Hyperspectral cube and bands visualization

### The hyperspectral VIS cube — a 3-D volume of 31 spectral bands

A hyperspectral image is a **three-dimensional cube**: two axes are the spatial dimensions of the image
(Image X, Image Y) and the third is a **spectral** axis representing the narrow wavelength bands. Each
VIS cube here is **1024 × 1024 × 31** — 31 bands sampled from 400 to 700 nm in 10 nm steps, where every
band is a 2-D image of the same face at one wavelength.

The figure below shows the cube with the spatial image on the **front face** and the **wavelength axis
receding into depth** (left), next to all 31 bands laid out individually (right). Reflectance changes
systematically with wavelength — the face is dark in the blue bands (≈400–470 nm) and brightens toward
red/near-IR, with the haemoglobin-sensitive bands (≈540–580 nm), from which the erythema index is
derived, in between. Shown for a disclosure subject.

In [ ]:
from mpl_toolkits.mplot3d import Axes3D, proj3d  # noqa: F401  (3-D projection + point projection)
import matplotlib.colors as mcolors

def visualize_cube(cube, wavelengths, ds=12, ncol=8, front_idx=None):
    """Iconic hyperspectral data-cube (left) + all 31 bands (right).

    Left: the front face is a representative spatial band (Image X x Image Y); the
    wavelength (spectral) axis recedes into depth, with the top and right faces showing
    the spectral variation. The "Wavelengths" label is aligned to the depth axis by
    projecting the axis endpoints to screen. Right: every band as its own 2-D image.
    """
    B = cube.shape[2]
    if front_idx is None:
        front_idx = B * 2 // 3          # representative mid-visible band (~600 nm)
    small = cube[::ds, ::ds, :].astype(float)
    hh, ww = small.shape[:2]

    # Create a Matplotlib engine scalar mapper (Just like imshow uses automatically)
    norm = mcolors.Normalize(vmin=small.min(), vmax=small.max())
    mapper = plt.cm.ScalarMappable(norm=norm, cmap="magma")

    fig = plt.figure(figsize=(22, 11))
    gs = fig.add_gridspec(1, 2, width_ratios=[1.0, 1.35], wspace=0.02)
    ax = fig.add_subplot(gs[0], projection="3d")
    cols, rows, bands = np.arange(ww), np.arange(hh), np.arange(B)

    # FRONT face (depth = 0): a representative band using native mapping
    Xf, Zf = np.meshgrid(cols, rows)
    front_data = np.flipud(small[:, :, front_idx])
    ax.plot_surface(Xf, np.zeros_like(Xf), Zf,
                    facecolors=mapper.to_rgba(front_data),
                    rstride=1, cstride=1, shade=False, antialiased=False)

    # TOP face — spectral depth using native mapping
    Xt, Yt = np.meshgrid(cols, bands)
    top_data = small[0, :, :].T
    ax.plot_surface(Xt, Yt, np.full(Xt.shape, hh - 1),
                    facecolors=mapper.to_rgba(top_data),
                    rstride=1, cstride=1, shade=False, antialiased=False)

    # RIGHT face — spectral depth using native mapping
    Yr, Zr = np.meshgrid(bands, rows)
    right_data = np.flipud(small[:, -1, :])
    ax.plot_surface(np.full(Yr.shape, ww - 1), Yr, Zr,
                    facecolors=mapper.to_rgba(right_data),
                    rstride=1, cstride=1, shade=False, antialiased=False)

    ax.set_xlabel("Image X axis", fontsize=10, labelpad=2)
    ax.set_ylabel(""); ax.set_zlabel("")   # placed as figure text below
    ax.set_xticks([]); ax.set_zticks([])
    yt = np.linspace(0, B - 1, 5).astype(int)
    ax.set_yticks(yt)
    ax.set_yticklabels([f"{int(wavelengths[i])} nm" for i in yt], fontsize=7)
    ax.set_box_aspect((1, 0.55, 1))
    ax.view_init(elev=16, azim=-60)
    ax.set_title("Hyperspectral Cube (Single Subject)", fontsize=13, pad=0)
    fig.text(0.16, 0.42, "Image Y axis", rotation="vertical",
             va="center", ha="center", fontsize=11)

    # Right: all 31 bands, each labelled by wavelength
    nrow = int(np.ceil(B / ncol))
    gs_r = gs[1].subgridspec(nrow, ncol, hspace=0.35, wspace=0.05)
    for i in range(nrow * ncol):
        a = fig.add_subplot(gs_r[i // ncol, i % ncol])
        if i < B:
            a.imshow(cube[:, :, i], cmap="magma")
            a.set_title(f"{int(wavelengths[i])} nm", fontsize=7)
        a.axis("off")
    fig.text(0.71, 0.90, "Individual Hyperspectral Bands", ha="center", fontsize=13)

    # "Wavelengths" label aligned to the depth axis by projecting its endpoints to screen
    fig.canvas.draw()

    def to_disp(x, y, z):
        xp, yp, _ = proj3d.proj_transform(x, y, z, ax.get_proj())
        return np.array(ax.transData.transform((xp, yp)))

    d0, d1 = to_disp(ww - 1, 0, 0), to_disp(ww - 1, B - 1, 0)
    vec = d1 - d0
    ang = np.degrees(np.arctan2(vec[1], vec[0]))
    perp = np.array([vec[1], -vec[0]]) / np.hypot(*vec)
    mid = (d0 + d1) / 2 + perp * 42
    fx, fy = fig.transFigure.inverted().transform(mid)
    fig.text(fx, fy, "Wavelengths", rotation=ang, rotation_mode="anchor",
             va="center", ha="center", fontsize=11)

    plt.show()

demo_subj = DISCLOSURE_SUBJECTS[0]
visualize_cube(cubes[demo_subj], get_wavelengths())


In [ ]:
wavelengths = get_wavelengths()
print(f"Wavelength array : {wavelengths}")
print(f"540 nm → band index {band_index_for_wavelength(540)}")

display_bands = [510, 540, 610]
fig, axes = plt.subplots(len(DISCLOSURE_SUBJECTS), len(display_bands),
                         figsize=(12, 4 * len(DISCLOSURE_SUBJECTS)))

for row, subj in enumerate(DISCLOSURE_SUBJECTS):
    for col, nm in enumerate(display_bands):
        idx = band_index_for_wavelength(nm)
        axes[row, col].imshow(cubes[subj][:, :, idx], cmap="gray")
        axes[row, col].set_title(f"{subj}  {nm} nm (band {idx})")
        axes[row, col].axis("off")

plt.suptitle("Single band slices — each panel is cube[:, :, idx]", y=1.01)
plt.tight_layout()
plt.show()

## 3. RGB vs hyperspectral alignment

Each row: RGB photo (left) alongside the 540 nm band (right). Check that faces are spatially aligned.

In [ ]:
idx_540 = band_index_for_wavelength(540)
fig, axes = plt.subplots(len(DISCLOSURE_SUBJECTS), 2,
                         figsize=(8, 4 * len(DISCLOSURE_SUBJECTS)))

for row, subj in enumerate(DISCLOSURE_SUBJECTS):
    axes[row, 0].imshow(rgbs[subj])
    axes[row, 0].set_title(f"{subj}  RGB")
    axes[row, 0].axis("off")
    axes[row, 1].imshow(cubes[subj][:, :, idx_540], cmap="gray")
    axes[row, 1].set_title(f"{subj}  540 nm band")
    axes[row, 1].axis("off")

plt.suptitle("RGB vs 540 nm — confirm spatial alignment", y=1.01)
plt.tight_layout()
plt.show()

## 4. Wavelength step confirmation

Verify that band spacing is truly 10 nm by inspecting the cube metadata (if available) or  
checking that the face appearance shifts predictably across adjacent bands.

In [ ]:
subj = "p012"
n_bands = config.CUBE_SHAPE[2]
fig, axes = plt.subplots(3, 11, figsize=(22, 6))
all_axes = axes.ravel()

for band_idx in range(n_bands):
    nm = config.WAVELENGTH_START_NM + band_idx * config.WAVELENGTH_STEP_NM
    all_axes[band_idx].imshow(cubes[subj][:, :, band_idx], cmap="gray")
    all_axes[band_idx].set_title(f"{nm}", fontsize=7)
    all_axes[band_idx].axis("off")

for ax in all_axes[n_bands:]:
    ax.axis("off")

plt.suptitle(f"{subj} — all 31 bands (400–700 nm). Gradual change confirms 10 nm step.")
plt.tight_layout()
plt.show()

print(f"Wavelengths: {list(get_wavelengths())}")

## 5. Build manifest and inspect

In [ ]:
df = build_manifest(config.DATA_ROOT, config.SPLIT_OVERRIDE)

print(f"Total rows     : {len(df)}  (expected 306)")
print(f"Unique subjects: {df['subject_id'].nunique()}  (expected 51)")
print()
print(df['split'].value_counts())
print()

# Verify overrides
for subj, expected in config.SPLIT_OVERRIDE.items():
    actual = df.loc[df['subject_id'] == subj, 'split'].unique()
    print(f"  Override {subj} → {expected}: {actual}")

df.head(10)